In [1]:
!pip install num2words
!pip install av
!pip install --upgrade transformers
!pip install opencv-python

In [2]:
import requests
import os
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import cv2
from PIL import Image
import random
import json
import re
from datetime import datetime, timezone
from pathlib import Path

c:\Users\Boop\anaconda3\envs\python3-9\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
cache_path = r"C:\Users\Boop\.cache\huggingface\hub\models--HuggingFaceTB--SmolVLM2-2.2B-Instruct\snapshots\482adb537c021c86670beed01cd58990d01e72e4"

processor = AutoProcessor.from_pretrained(cache_path)
print("Processor loaded!")

✅ Processor loaded!


In [4]:
model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
).to("cuda")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.96s/it]


In [5]:
#load relative path to video_25.mp4
relative_path = os.path.join("..", "Moment-Retrieval-and-Video-Summarization", "TVSum", "video_25.mp4")
video_path = os.path.abspath(relative_path)
print(video_path)
print("File exists:", os.path.exists(video_path))

c:\Users\Boop\Workspace\ComputerVision\Moment-Retrieval-and-Video-Summarization\TVSum\video_25.mp4
File exists: True


In [6]:
def save_generated_description(generated_text, file_path="generated_descriptions_trials.txt"):

    file_path = Path(file_path)

    text = generated_text[0] if isinstance(generated_text, list) else str(generated_text)

    # Split assistant response
    assistant_marker = "Assistant:"
    assistant_idx = text.rfind(assistant_marker)

    if assistant_idx != -1:
        prompt_part = text[:assistant_idx].strip()
        assistant_response = text[assistant_idx + len(assistant_marker):].strip()
    else:
        prompt_part = text.strip()
        assistant_response = ""

    # Try to extract only the prompt after the last "Frame from xx:xx\n\n"
    frame_matches = list(re.finditer(r":\n\n", prompt_part))
    if frame_matches:
        last_match = frame_matches[-1]
        prompt = prompt_part[last_match.end():].strip()
    else:
        prompt = prompt_part

    # Count frames analysed
    frames_analysed = len(re.findall(r"\nFrame from ", text))

    record = {
        "timestamp_saved": datetime.now(timezone.utc).isoformat(),
        "frames_analysed": frames_analysed,
        "prompt": prompt,
        "assistant_response": assistant_response        
    }

    with file_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n\n")

    return record


def load_generated_descriptions(file_path="generated_descriptions_trials.txt"):
    """
    Load all saved description records from the JSONL file.
    """
    file_path = Path(file_path)
    if not file_path.exists():
        return []

    records = []
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

In [7]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "video", "path": video_path},
            {"type": "text", "text": """Describe what is happening in the video, sequentially focusing on the most important events. 
             Describe the events in detail, including the people involved, their appearance, their actions, and any relevant context. 
             Avoid generic descriptions and focus on specific moments that stand out. 
             List the events in order of occurrence."""}
        ]
    },
    
]

inputs = processor.apply_chat_template(
    messages,
    num_frames=100,  # more frames = better coverage
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

generated_ids = model.generate(
    **inputs,
    do_sample=False,
    max_new_tokens=512,
    repetition_penalty=1.3,      # penalises repeating the same phrases
    no_repeat_ngram_size=4,      # forbids repeating any 4-word sequence
)

generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
print(generated_texts[0])
save_generated_description(generated_texts[0])

c:\Users\Boop\anaconda3\envs\python3-9\lib\site-packages\transformers\video_processing_utils.py:879: UserWarning: `torchcodec` is not installed and cannot be used to decode the video by default. Falling back to `torchvision`. Note that `torchvision` decoding is deprecated and will be removed in future versions. 
  warnings.warn(
c:\Users\Boop\anaconda3\envs\python3-9\lib\site-packages\transformers\video_utils.py:524: UserWarning: Using `torchvision` for video decoding is deprecated and will be removed in future versions. Please use `torchcodec` instead.
  warnings.warn(
c:\Users\Boop\anaconda3\envs\python3-9\lib\site-packages\torchvision\io\_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.war

User: You are provided the following series of one hundred frames from a 0:04:34 [H:MM:SS] video.

Frame from 00:01:
Frame from 00:03:
Frame from 00:06:
Frame from 00:09:
Frame from 00:11:
Frame from 00:14:
Frame from 00:17:
Frame from 00:20:
Frame from 00:22:
Frame from 00:25:
Frame from 00:28:
Frame from 00:31:
Frame from 00:33:
Frame from 00:36:
Frame from 00:39:
Frame from 00:42:
Frame from 00:44:
Frame from 00:47:
Frame from 00:50:
Frame from 00:53:
Frame from 00:55:
Frame from 00:58:
Frame from 01:01:
Frame from 01:04:
Frame from 01:06:
Frame from 01:09:
Frame from 01:12:
Frame from 01:15:
Frame from 01:17:
Frame from 01:20:
Frame from 01:23:
Frame from 01:26:
Frame from 01:28:
Frame from 01:31:
Frame from 01:34:
Frame from 01:37:
Frame from 01:39:
Frame from 01:42:
Frame from 01:45:
Frame from 01:48:
Frame from 01:50:
Frame from 01:53:
Frame from 01:56:
Frame from 01:59:
Frame from 02:01:
Frame from 02:04:
Frame from 02:07:
Frame from 02:10:
Frame from 02:12:
Frame from 02:15:
F

{'timestamp_saved': '2026-05-11T00:13:04.648228+00:00',
 'frames_analysed': 100,
 'prompt': 'Describe what is happening in the video, sequentially focusing on the most important events. \n             Describe the events in detail, including the people involved, their appearance, their actions, and any relevant context. \n             Avoid generic descriptions and focus on specific moments that stand out. \n             List the events in order of occurrence.',
 'assistant_response': "The video starts with an aerial view of a cityscape featuring tall buildings under overcast skies. A person wearing headphones takes photos while standing near graffiti-covered walls. They then walk through crowded streets before entering a building where they take more pictures inside. Next to them sits another individual holding up his camera for someone else's photo. Inside, two individuals sit at tables covered by red Coca Cola cans; later, we see this same table set outside against a backdrop of blu

In [8]:
def save_generated_annotation(events, prompt, file_path="generated_annotations_trials.txt"):

    file_path = Path(file_path)

    text = "\n".join(events) if isinstance(events, list) else str(events)

    record = {
        "timestamp_saved": datetime.now(timezone.utc).isoformat(),
        "prompt": prompt,
        "annotation": text
    }
    with file_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n\n")

    return record

In [9]:
def extract_events_with_ollama(prompt, model="llama3.2"):

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False}
    )
    
    return response.json()["response"]


def parse_salient_events(llm_output):
    events = []
    for line in llm_output.strip().splitlines():
        if not line.strip():
            continue

        # If line starts with "Salient event" keep text after first colon
        if ":" in line and line.lower().startswith("salient event"):
            description = line.split(":", 1)[1].strip()
        else:
            # remove leading "Salient event X" if present (with optional punctuation)
            description = re.sub(r"^\s*Salient event\s*\d+\s*[:\.\)\-]*\s*", "", line, flags=re.I)

            # remove leading numeric list markers like "1. ", "1) ", "1 - "
            description = re.sub(r"^\s*\d+\s*[:\.\)\-]*\s*", "", description)

        if description:
            events.append(description)
    return events

In [10]:
vlm_description = """The video starts with a person walking down a street in a city, passing by various buildings and shops. The scene then transitions to a person sitting on the ground, wearing earphones and looking at something in their hands. The next scene shows a group of people gathered around a banner, with one person holding a microphone. The video then shows a group of people standing on a beach, with a large building in the background. The final scene shows a person lying on a bed, holding a camera."""
prompt = f"""You are an expert video annotator. Below is a description of a video.
Understand the overall context and narrative of the video based on the description provided.
Extract between 4 and 10 salient events from this description.

A salient event is important for understanding the overall narrative, clearly distinguishable, and described in 1-2 concise sentences. Don't narrate the flow of the video, just describe the events.

Format EXACTLY like this:
Salient event 1: [description]
Salient event 2: [description]
...
Only output the numbered list, nothing else.

Video description:
{vlm_description}"""

In [11]:
result = extract_events_with_ollama(prompt)
print(result)

events = parse_salient_events(result)
print("\n--- Parsed Events ---")
for i, event in enumerate(events, 1):
    print(f"{event}")

save_generated_annotation(events, prompt=prompt)

1. A person walks down a city street, passing by various buildings and shops.
2. A person sits on the ground, wearing earphones and looking at something in their hands.
3. A group of people gather around a banner, with one person holding a microphone.
4. A group of people stand on a beach, with a large building in the background.
5. A person lies on a bed, holding a camera.

--- Parsed Events ---
A person walks down a city street, passing by various buildings and shops.
A person sits on the ground, wearing earphones and looking at something in their hands.
A group of people gather around a banner, with one person holding a microphone.
A group of people stand on a beach, with a large building in the background.
A person lies on a bed, holding a camera.


{'timestamp_saved': '2026-05-11T00:13:14.465499+00:00',
 'prompt': "You are an expert video annotator. Below is a description of a video.\nUnderstand the overall context and narrative of the video based on the description provided.\nExtract between 4 and 10 salient events from this description.\n\nA salient event is important for understanding the overall narrative, clearly distinguishable, and described in 1-2 concise sentences. Don't narrate the flow of the video, just describe the events.\n\nFormat EXACTLY like this:\nSalient event 1: [description]\nSalient event 2: [description]\n...\nOnly output the numbered list, nothing else.\n\nVideo description:\nThe video starts with a person walking down a street in a city, passing by various buildings and shops. The scene then transitions to a person sitting on the ground, wearing earphones and looking at something in their hands. The next scene shows a group of people gathered around a banner, with one person holding a microphone. The vide

In [12]:
save_generated_annotation(events, prompt=prompt)

{'timestamp_saved': '2026-05-11T00:17:18.416249+00:00',
 'prompt': "You are an expert video annotator. Below is a description of a video.\nUnderstand the overall context and narrative of the video based on the description provided.\nExtract between 4 and 10 salient events from this description.\n\nA salient event is important for understanding the overall narrative, clearly distinguishable, and described in 1-2 concise sentences. Don't narrate the flow of the video, just describe the events.\n\nFormat EXACTLY like this:\nSalient event 1: [description]\nSalient event 2: [description]\n...\nOnly output the numbered list, nothing else.\n\nVideo description:\nThe video starts with a person walking down a street in a city, passing by various buildings and shops. The scene then transitions to a person sitting on the ground, wearing earphones and looking at something in their hands. The next scene shows a group of people gathered around a banner, with one person holding a microphone. The vide